# Grouped Query Attention (GQA) in the 50M model

This notebook instantiates the project model with GQA. It keeps 8 *query* heads and uses 2 shared *key/value* heads: each group of 4 query heads shares K and V.

In [ ]:
from copy import deepcopy

import torch

from llm_mini_lab.models.gpt import GPTModel
from llm_mini_lab.training.core import GPT_CONFIG_50M

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:
# Fair comparison: the rest of the architecture stays unchanged.
mha_cfg = deepcopy(GPT_CONFIG_50M)
mha_cfg["n_kv_heads"] = mha_cfg["n_heads"]

gqa_cfg = deepcopy(GPT_CONFIG_50M)
gqa_cfg["n_kv_heads"] = 2

mha = GPTModel(mha_cfg)
gqa = GPTModel(gqa_cfg)

mha_params = count_parameters(mha)
gqa_params = count_parameters(gqa)
print(f"MHA: {mha_params:,} parameters")
print(f"GQA: {gqa_params:,} parameters")
print(f"Saved: {mha_params - gqa_params:,} ({100 * (mha_params - gqa_params) / mha_params:.2f}%)")


In [ ]:
# Verify one forward pass.
tokens = torch.randint(0, gqa_cfg["vocab_size"], (2, 32))
logits = gqa(tokens)

assert logits.shape == (2, 32, gqa_cfg["vocab_size"])
first_attention = gqa.trf_blocks[0].att
print("Logits shape:", tuple(logits.shape))
print(f"Query heads: {first_attention.num_heads}; shared K/V heads: {first_attention.num_kv_heads}")
print(f"Each K/V head serves {first_attention.kv_group_size} query heads.")


## Using it in training

`GPT_CONFIG_50M` already sets `n_kv_heads: 2`, so scripts that import this configuration will use GQA automatically. To return to MHA, remove this key or set it to `n_heads`. For GQA, `n_kv_heads` must divide `n_heads` exactly.

The memory reduction becomes especially meaningful with a K/V cache during generation: only the 2 K/V heads need to be stored, rather than all 8 MHA heads.